# 面试问题：梯度消失与爆炸为什么发生，怎样诊断并用初始化、残差和裁剪缓解？

**一句话回答**：反向梯度是逐层 Jacobian 的连乘；奇异值长期小于 1 会消失，大于 1 会爆炸，饱和激活又会把局部导数压到接近 0。诊断要记录各层 activation/gradient norm、非有限值和更新比；缓解包括 Xavier/He 初始化、非饱和激活、归一化、残差路径、合理深度/学习率和 global-norm clipping。

本 Notebook 用 PyTorch 基础算子构造可证明的标量链、深 MLP、残差网络和手写 global-norm clipping。

In [ ]:
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

SEED96=9601; torch.manual_seed(SEED96); np.random.seed(SEED96)  # 计算并保存当前步骤的中间状态。
assert SEED96==9601  # 用受控断言验证关键不变量。
assert torch.__version__  # 用受控断言验证关键不变量。
assert math.prod([.5]*4)==.0625  # 用受控断言验证关键不变量。

## 1. Jacobian 连乘的最小反例

标量链 `h_L=w^L x` 对输入梯度是 `w^L`。深度 30 时 w=0.5 几乎为 0，w=1.5 已很大。这不是优化器造成，而是链式法则本身。矩阵网络对应 Jacobian 奇异值的连乘，方向之间还可能不同。

用 autograd 与解析式交叉验证。

In [ ]:
def scalar_chain96(weight,depth):  # 定义本节可复用的核心函数。
    x=torch.tensor(1.,requires_grad=True); h=x  # 计算并保存当前步骤的中间状态。
    for _ in range(depth): h=h*weight  # 遍历输入元素以累积或检查结果。
    h.backward(); return float(x.grad),float(h.detach())  # 执行当前语句以推进本节示例。
vanish_grad96,_=scalar_chain96(.5,30); explode_grad96,_=scalar_chain96(1.5,30)  # 计算并保存当前步骤的中间状态。
assert math.isclose(vanish_grad96,.5**30,rel_tol=1e-6)  # 用受控断言验证关键不变量。
assert math.isclose(explode_grad96,1.5**30,rel_tol=1e-5)  # 用受控断言验证关键不变量。
assert vanish_grad96<1e-8 and explode_grad96>1e5  # 用受控断言验证关键不变量。

## 2. 饱和激活会再乘一个小导数

sigmoid 导数 `s(1-s)` 最大只有 0.25，输入绝对值很大时接近 0；tanh 也会饱和。ReLU 正区间导数为 1，但负区间为 0，可能产生 dead neuron；GELU/SiLU 更平滑但仍不消除矩阵尺度问题。

诊断同时看 pre-activation 分布和 local derivative。

In [ ]:
z96=torch.tensor([-10.,-2.,0.,2.,10.],requires_grad=True); sig96=torch.sigmoid(z96); sig96.sum().backward(); derivative96=z96.grad.detach()  # 计算并保存当前步骤的中间状态。
assert math.isclose(float(derivative96[2]),.25,rel_tol=1e-6)  # 用受控断言验证关键不变量。
assert derivative96[0]<1e-4 and derivative96[-1]<1e-4  # 用受控断言验证关键不变量。
relu_z96=torch.tensor([-1.,1.],requires_grad=True); torch.relu(relu_z96).sum().backward()  # 计算并保存当前步骤的中间状态。
assert relu_z96.grad.tolist()==[0.,1.]  # 用受控断言验证关键不变量。

## 3. Xavier 与 He 初始化从公式实现

Xavier 令方差约 `2/(fan_in+fan_out)`，常用于 tanh/线性；He 令方差约 `2/fan_in`，补偿 ReLU 丢掉一半激活。初始化目标是前后向方差在深层不过快漂移，不保证训练一定成功。

下面不用 `nn.init`，直接采样并验证经验标准差。

In [ ]:
def init_weight96(shape,kind,generator):  # 定义本节可复用的核心函数。
    fan_out,fan_in=shape  # 计算并保存当前步骤的中间状态。
    if kind=="xavier": std=math.sqrt(2/(fan_in+fan_out))  # 按当前条件选择后续控制路径。
    elif kind=="he": std=math.sqrt(2/fan_in)  # 按当前条件选择后续控制路径。
    else: raise ValueError("init_contract")  # 执行当前语句以推进本节示例。
    return torch.randn(shape,generator=generator)*std,std  # 返回当前分支计算出的结果。
gen96=torch.Generator().manual_seed(96); wx96,sx96=init_weight96((512,256),"xavier",gen96); wh96,sh96=init_weight96((512,256),"he",gen96)  # 计算并保存当前步骤的中间状态。
assert abs(float(wx96.std())-sx96)/sx96<.02  # 用受控断言验证关键不变量。
assert abs(float(wh96.std())-sh96)/sh96<.02  # 用受控断言验证关键不变量。
assert sh96>sx96  # 用受控断言验证关键不变量。

## 4. 深 MLP 的 activation/gradient trace

手写 `DeepMLP96`，每层保存 activation；反向后读取 parameter grad norm。健康并不意味着所有层完全相等，但不应跨层指数级归零/爆炸。日志按 layer index、dtype、step 输出，避免只看总梯度。

使用 He+ReLU 与故意过大初始化对比。

In [ ]:
class DeepMLP96(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,depth,width,std):  # 定义本节可复用的核心函数。
        super().__init__(); self.layers=nn.ModuleList([nn.Linear(width,width,bias=False) for _ in range(depth)]); self.activations=[]  # 计算并保存当前步骤的中间状态。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            for layer in self.layers: layer.weight.normal_(0,std)  # 遍历输入元素以累积或检查结果。
    def forward(self,x):  # 定义本节可复用的核心函数。
        self.activations=[]  # 计算并保存当前步骤的中间状态。
        for layer in self.layers: x=torch.relu(layer(x)); self.activations.append(x)  # 遍历输入元素以累积或检查结果。
        return x  # 返回当前分支计算出的结果。
def trace96(std):  # 定义本节可复用的核心函数。
    torch.manual_seed(9610); m=DeepMLP96(16,32,std); x=torch.randn(64,32); loss=m(x).square().mean(); loss.backward(); return m,[float(a.detach().std()) for a in m.activations],[float(l.weight.grad.norm()) for l in m.layers]  # 计算并保存当前步骤的中间状态。
healthy96,act_h96,grad_h96=trace96(math.sqrt(2/32)); huge96,act_x96,grad_x96=trace96(.8)  # 计算并保存当前步骤的中间状态。
assert all(math.isfinite(x) for x in act_h96+grad_h96)  # 用受控断言验证关键不变量。
assert max(act_h96)<5 and min(grad_h96)>0  # 用受控断言验证关键不变量。
assert max(act_x96)>max(act_h96)*100 or max(grad_x96)>max(grad_h96)*100  # 用受控断言验证关键不变量。

## 5. 残差连接提供恒等梯度路径

纯链 `h=F(h)` 的梯度必须穿过每个 F；残差 `h=h+F(h)` 的 Jacobian 是 `I+J_F`，即使 F 分支较小，仍有恒等路径。它缓解但不彻底消除爆炸，深网络仍需初始化/Norm/缩放。

用相同小权重比较输入梯度。

In [ ]:
def matrix_chain_grad96(residual,depth=20):  # 定义本节可复用的核心函数。
    torch.manual_seed(9620); x=torch.randn(1,16,requires_grad=True); h=x; weights=[torch.randn(16,16)*.03 for _ in range(depth)]  # 计算并保存当前步骤的中间状态。
    for w in weights:  # 遍历输入元素以累积或检查结果。
        branch=torch.tanh(h@w); h=h+branch if residual else branch  # 计算并保存当前步骤的中间状态。
    h.sum().backward(); return float(x.grad.norm())  # 执行当前语句以推进本节示例。
plain_chain96=matrix_chain_grad96(False); residual_chain96=matrix_chain_grad96(True)  # 计算并保存当前步骤的中间状态。
assert residual_chain96>plain_chain96*1e6  # 用受控断言验证关键不变量。
assert residual_chain96>1 and plain_chain96<1e-6  # 用受控断言验证关键不变量。
assert math.isfinite(residual_chain96)  # 用受控断言验证关键不变量。

## 6. 手写 global-norm clipping

先把所有 parameter gradient 的平方和求根得到 global norm；若超过 max_norm，所有梯度乘同一系数 `max_norm/(norm+eps)`，保持方向。逐参数分别裁剪会改变方向。AMP 下必须先 unscale 再判断/裁剪。

clipping 是保险丝，不应掩盖持续爆炸的根因。

In [ ]:
def clip_global_norm96(parameters,max_norm,eps=1e-12):  # 定义本节可复用的核心函数。
    grads=[p.grad for p in parameters if p.grad is not None]  # 计算并保存当前步骤的中间状态。
    if max_norm<=0 or not grads: raise ValueError("clip_contract")  # 按当前条件选择后续控制路径。
    total=torch.sqrt(sum(g.float().square().sum() for g in grads)); coef=min(1.,max_norm/(float(total)+eps))  # 计算并保存当前步骤的中间状态。
    for g in grads: g.mul_(coef)  # 遍历输入元素以累积或检查结果。
    return float(total),coef  # 返回当前分支计算出的结果。
params96=list(huge96.parameters()); before96,coef96=clip_global_norm96(params96,1.)  # 计算并保存当前步骤的中间状态。
after96=math.sqrt(sum(float(p.grad.float().square().sum()) for p in params96 if p.grad is not None))  # 计算并保存当前步骤的中间状态。
assert before96>1 and 0<coef96<1  # 用受控断言验证关键不变量。
assert after96<=1.00001  # 用受控断言验证关键不变量。
assert all(torch.isfinite(p.grad).all() for p in params96)  # 用受控断言验证关键不变量。

## 7. 更新比、非有限检测与诊断顺序

记录 `||grad||/||param||` 和 `||lr*grad||/||param||`，区分梯度尺度与学习率问题。loss NaN 时先找第一个非有限 activation/gradient，再检查数据、mask、loss scaling 和 optimizer state；只把学习率除 10 可能暂时掩盖 bug。

任何 NaN/Inf 都应跳过 optimizer step，并触发样本/step trace。

In [ ]:
def update_ratios96(model,lr):  # 定义本节可复用的核心函数。
    out=[]  # 计算并保存当前步骤的中间状态。
    for name,p in model.named_parameters():  # 遍历输入元素以累积或检查结果。
        if p.grad is not None:  # 按当前条件选择后续控制路径。
            pn=float(p.detach().norm()); gn=float(p.grad.detach().norm()); out.append((name,gn/(pn+1e-12),lr*gn/(pn+1e-12),bool(torch.isfinite(p.grad).all())))  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。
ratios96=update_ratios96(healthy96,1e-3)  # 计算并保存当前步骤的中间状态。
assert len(ratios96)==16 and all(r[3] for r in ratios96)  # 用受控断言验证关键不变量。
assert all(r[1]>=0 and r[2]>=0 for r in ratios96)  # 用受控断言验证关键不变量。
assert all(math.isclose(r[2],r[1]*1e-3,rel_tol=1e-6) for r in ratios96)  # 用受控断言验证关键不变量。

## 8. 训练门禁与可复现制品

发布训练 recipe 需要记录初始化、激活、Norm、残差缩放、clip norm、dtype/loss scale、optimizer/lr 和 gradient accumulation。回归用固定 mini-batch 比较逐层 trace 分位数；架构加深后不能只看最终 loss。

告警阈值按模型/层类型校准，embedding 稀疏梯度与 dense block 不应共用同一绝对阈值。

In [ ]:
manifest96={"schema":1,"network":"deep_relu_mlp","depth":16,"width":32,"init":"he_normal","residual":"recommended","clip_global_norm":1.,"diagnostics":["activation_std","grad_norm","update_ratio","nonfinite"]}; digest96=hashlib.sha256(json.dumps(manifest96,sort_keys=True,separators=(",",":")).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(digest96)==64 and manifest96["depth"]==len(healthy96.layers)  # 用受控断言验证关键不变量。
assert manifest96["clip_global_norm"]==1.  # 用受控断言验证关键不变量。
assert set(manifest96["diagnostics"])=={"activation_std","grad_norm","update_ratio","nonfinite"}  # 用受控断言验证关键不变量。
print({"vanish":vanish_grad96,"explode":round(explode_grad96,1),"plain_chain":plain_chain96,"residual_chain":residual_chain96,"clip_before":before96})  # 执行当前语句以推进本节示例。

## 9. 面试收束、参考与练习

回答闭环：Jacobian 连乘 → 激活饱和 → Xavier/He → 逐层 trace → 残差恒等路径 → global clipping → update ratio/NaN 定位 → recipe 版本。不要把“加 BatchNorm、调小学习率”当成无条件答案。

练习：测 Jacobian 最大奇异值；实现 LeakyReLU；比较 Pre-Norm/Post-Norm 深度；在 AMP unscale 前后错误裁剪并观察差异；加入 gradient checkpointing 验证数值一致。

参考：[Xavier 初始化](https://proceedings.mlr.press/v9/glorot10a.html)、[He 初始化](https://arxiv.org/abs/1502.01852)、[Deep Residual Learning](https://arxiv.org/abs/1512.03385)。